In [1]:
import requests
from bs4 import BeautifulSoup

In [2]:
BASE_SITE = "http://books.toscrape.com/"
HEADERS = {"User-Agent": "Mozilla/5.0"}

In [3]:
response = requests.get(BASE_SITE, headers=HEADERS)
soup = BeautifulSoup(response.text, "html.parser")

In [4]:
categories = {}

# Select only the inner list of genres (not Books link)
genre_links = soup.select("ul.nav-list ul li a")

for a in genre_links:
    genre_name = a.text.strip()
    genre_url = BASE_SITE + a["href"]
    categories[genre_name] = genre_url

In [5]:
print(f"Total genres found: {len(categories)}")

Total genres found: 50


In [6]:
list(categories.items())

[('Travel',
  'http://books.toscrape.com/catalogue/category/books/travel_2/index.html'),
 ('Mystery',
  'http://books.toscrape.com/catalogue/category/books/mystery_3/index.html'),
 ('Historical Fiction',
  'http://books.toscrape.com/catalogue/category/books/historical-fiction_4/index.html'),
 ('Sequential Art',
  'http://books.toscrape.com/catalogue/category/books/sequential-art_5/index.html'),
 ('Classics',
  'http://books.toscrape.com/catalogue/category/books/classics_6/index.html'),
 ('Philosophy',
  'http://books.toscrape.com/catalogue/category/books/philosophy_7/index.html'),
 ('Romance',
  'http://books.toscrape.com/catalogue/category/books/romance_8/index.html'),
 ('Womens Fiction',
  'http://books.toscrape.com/catalogue/category/books/womens-fiction_9/index.html'),
 ('Fiction',
  'http://books.toscrape.com/catalogue/category/books/fiction_10/index.html'),
 ('Childrens',
  'http://books.toscrape.com/catalogue/category/books/childrens_11/index.html'),
 ('Religion',
  'http://book

In [7]:
import pandas as pd
import time

In [8]:
# Container for all books across all genres
books_data = []
# safety limit
MAX_PAGES = 20  

In [9]:
for genre, genre_url in categories.items():

    # Skip non-genre pages
    if genre.lower() in ["add a comment", "default"]:
        continue

    print(f"Scraping genre: {genre}")

    for page_num in range(1, MAX_PAGES + 1):

        #Correct handling of first page
        if page_num == 1:
            page_url = genre_url
        else:
            page_url = genre_url.replace(
                "index.html", f"page-{page_num}.html"
            )

        try:
            response = requests.get(
                page_url,
                headers=HEADERS,
                timeout=10   
            )
        except requests.exceptions.RequestException:
            break  # stop this genre if request fails

        soup = BeautifulSoup(response.text, "html.parser")
        books = soup.select("article.product_pod")

        # Stop when no books are found
        if not books:
            break

        for book in books:
            books_data.append({
                "title": book.h3.a["title"],
                "price": book.select_one("p.price_color").text,
                "availability": book.select_one("p.instock.availability").text.strip(),
                "rating_word": book.select_one("p.star-rating")["class"][1],
                "genre": genre
            })

        time.sleep(0.5)  

Scraping genre: Travel
Scraping genre: Mystery
Scraping genre: Historical Fiction
Scraping genre: Sequential Art
Scraping genre: Classics
Scraping genre: Philosophy
Scraping genre: Romance
Scraping genre: Womens Fiction
Scraping genre: Fiction
Scraping genre: Childrens
Scraping genre: Religion
Scraping genre: Nonfiction
Scraping genre: Music
Scraping genre: Science Fiction
Scraping genre: Sports and Games
Scraping genre: Fantasy
Scraping genre: New Adult
Scraping genre: Young Adult
Scraping genre: Science
Scraping genre: Poetry
Scraping genre: Paranormal
Scraping genre: Art
Scraping genre: Psychology
Scraping genre: Autobiography
Scraping genre: Parenting
Scraping genre: Adult Fiction
Scraping genre: Humor
Scraping genre: Horror
Scraping genre: History
Scraping genre: Food and Drink
Scraping genre: Christian Fiction
Scraping genre: Business
Scraping genre: Biography
Scraping genre: Thriller
Scraping genre: Contemporary
Scraping genre: Spirituality
Scraping genre: Academic
Scraping genr

In [10]:
raw_books_df = pd.DataFrame(books_data)

In [11]:
raw_books_df.to_csv("../data/raw_books_with_genre.csv", index=False)

In [12]:
print(raw_books_df.shape)

(781, 5)


In [13]:
raw_books_df.head()

,title,price,availability,rating_word,genre
0,It's Only the Himalayas,Â£45.17,In stock,Two,Travel
1,Full Moon over Noahâs Ark: An Odyssey to Mou...,Â£49.43,In stock,Four,Travel
2,See America: A Celebration of Our National Par...,Â£48.87,In stock,Three,Travel
3,Vagabonding: An Uncommon Guide to the Art of L...,Â£36.94,In stock,Two,Travel
4,Under the Tuscan Sun,Â£37.33,In stock,Three,Travel
